# 4.2 — Guided project: Learning-centre analysis

**Concrete question:** For January–June 2026, which course has the higher overall completion rate, and which has the lower material cost per completion: Python Foundations or Digital Skills? Quantify both differences, decide whether these 24 records justify ranking the courses, and propose one item of data needed next.

## 1. File location and definition of done

The file is `learning-centres-practice.csv`. Download it directly from the Moodle page “4.2 Dataset and project brief.” In Python Lab it is available at `/home/jovyan/work/data/learning-centres-practice.csv`; the next cell locates it without depending on the start folder.

Immediately after loading, it has **24 rows and 10 columns**. Courses are `Python Foundations` and `Digital Skills`, covering `2026-01` through `2026-06`. Your final notebook must contain a quality audit, course summary, reconciliation, one primary chart, and a 150–250 word answer.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_course_data(filename):
    roots=[Path.cwd(),*Path.cwd().parents,Path.home()/"work",Path("/opt/python-lab/course-materials")]
    checked=[]
    for root in roots:
        for p in (root/"data"/filename,root/filename):
            if p in checked: continue
            checked.append(p)
            if p.is_file(): return p
    raise FileNotFoundError("Course data was not found:\n"+"\n".join(map(str,checked)))

data_file=find_course_data("learning-centres-practice.csv")
raw=pd.read_csv(data_file)
print("Source:",data_file.resolve(),"shape:",raw.shape)


## Confirm the exact file that opened

The next output must show the absolute path, `(24, 10)`, ten column names, and the first five rows. Stop and check the file if the shape differs.

In [ ]:
print(data_file.resolve())
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())
assert raw.shape == (24, 10)


## 2. Inspect these ten columns

Confirm `month`, `centre_id`, `centre_name`, `district`, `course`, `registered`, `attended`, `completed`, `training_hours`, and `material_cost`. Then display types, missing count per column, district labels, and minimum and maximum for counts, hours, and cost. Do not change values yet.

In [ ]:
required={"month","centre_id","district","course","registered","attended","completed","material_cost"}
missing=required-set(raw.columns)
if missing: raise KeyError(sorted(missing))
print(raw.dtypes); print(raw.isna().sum()); print(sorted(raw["district"].dropna().unique()))


## 3. Run three separate quality checks

Count three named Boolean masks separately: (1) missing `attended`; (2) `completed > attended` where both are known; and (3) duplicate `centre_id + month + course`. Preserve `district_raw` while normalising whitespace and case. Do not guess learner counts; set problem rows to `analysis_ready=False`. The audit table must record issue, rule, affected count, and action.

In [ ]:
clean=raw.copy(); clean["district_raw"]=clean["district"]; clean["district"]=clean["district"].astype("string").str.strip().str.title()
missing_attended=clean["attended"].isna(); invalid_completion=clean["completed"].notna() & clean["attended"].notna() & (clean["completed"]>clean["attended"])
key=["centre_id","month","course"]; duplicate=clean.duplicated(key,keep=False)
clean["analysis_ready"]=~(missing_attended|invalid_completion|duplicate)
analysis=clean.loc[clean["analysis_ready"]].copy()
audit=pd.DataFrame({"issue":["missing attended","completion above attendance","duplicate business key"],"affected":[int(missing_attended.sum()),int(invalid_completion.sum()),int(duplicate.sum())],"action":["flag; exclude from rate analysis","flag; do not guess","review duplicate group"]})
audit


## 4. Complete this course summary

`summary` must contain `course`, `records`, `centres`, `registered`, `completed`, `material_cost`, `completion_rate`, and `cost_per_completion`. Calculate completion as total completed / total registered × 100 and cost per completion as total cost / total completed. Do not average row rates or row costs. Calculate the course completion difference in percentage points and the cost difference in the cost unit.

In [ ]:
summary=analysis.groupby("course").agg(records=("centre_id","size"),centres=("centre_id","nunique"),registered=("registered","sum"),completed=("completed","sum"),material_cost=("material_cost","sum")).reset_index()
summary["completion_rate"]=summary["completed"]/summary["registered"]*100
summary["cost_per_completion"]=summary["material_cost"]/summary["completed"]
summary.round(2)


## 5. Reconcile counts and totals

Use assertions to verify source equals analysis-ready plus flagged rows and grouped totals equal analysis detail.

In [ ]:
assert len(raw)==int(clean["analysis_ready"].sum())+int((~clean["analysis_ready"]).sum())
assert summary["registered"].sum()==analysis["registered"].sum()
assert summary["completed"].sum()==analysis["completed"].sum()
assert abs(summary["material_cost"].sum()-analysis["material_cost"].sum())<1e-9
print("Validation passed")


## 6. Create one primary chart for one question

Compare overall completion rates with horizontal bars on a zero-to-100% axis. Report cost per completion in the table and use a second chart only if needed. Label axes, units, title, and n.

In [ ]:
plot_data=summary.sort_values("completion_rate")
fig,ax=plt.subplots(figsize=(7,4)); ax.barh(plot_data["course"],plot_data["completion_rate"],color="#356a9a"); ax.set(xlabel="Overall completion rate (%)",ylabel="Course",title=f"Completion by course (analysis-ready centre-months n={len(analysis)})"); ax.set_xlim(0,100); ax.grid(axis="x",alpha=.25); plt.tight_layout(); plt.show()


## 7. Answer these five prompts in order

In 150–250 words state: (1) both overall completion rates and the percentage-point difference; (2) both costs per completion and their difference; (3) which is higher/lower without saying one course is superior; (4) analysis-ready row count, period, and at least one limitation; and (5) one additional item of data needed to investigate performance or causes.

In [ ]:
# Write the report here.


## Pre-submission checklist

- Run All completes without errors.
- Loaded path and `(24, 10)` are shown.
- Three quality issues are counted separately and an audit table is present.
- `summary` has the specified eight columns and two courses.
- Row, registration, completion, and cost reconciliations pass.
- The primary chart has a 0–100% scale, title, axis name, unit, and analysis-ready n.
- The report answers all five prompts.